# State Space Models (S4 & Mamba)

Companion notebook for the [State Space Models lesson](https://ml-viz.vercel.app/courses/rnns/04-state-space-models).

We implement a linear SSM both ways — as a **step-by-step recurrence** and as a **convolution** with
the SSM kernel — and verify they give *identical* outputs. Then we show the **O(L) vs O(L²)** scaling
advantage over attention. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## 1 — The linear recurrence (inference mode)

h_t = A h_{t-1} + B u_t,  y_t = C h_t. We roll it forward one step at a time — O(1) state update per
token, like an RNN but with a *linear* transition.

In [ ]:
# scalar state-space system for clarity (state dim 1)
A, B, C = 0.8, 0.5, 1.3
u = rng.normal(size=12)

def ssm_recurrent(A, B, C, u):
    h, ys = 0.0, []
    for ut in u:
        h = A * h + B * ut        # O(1) per step
        ys.append(C * h)
    return np.array(ys)

y_rec = ssm_recurrent(A, B, C, u)
print('recurrent output:', y_rec[:5].round(3), '...')

## 2 — The same thing as a convolution (training mode)

Because the recurrence is linear, y = K * u where the kernel is K_j = C·A^j·B. Computing the kernel
once and convolving processes the whole sequence in parallel — no step loop.

In [ ]:
def ssm_kernel(A, B, C, L):
    return np.array([C * (A ** j) * B for j in range(L)])

def ssm_conv(A, B, C, u):
    L = len(u)
    K = ssm_kernel(A, B, C, L)
    # causal convolution: y_t = sum_{j<=t} K_j u_{t-j}
    return np.array([np.dot(K[:t+1], u[t::-1]) for t in range(L)])

y_conv = ssm_conv(A, B, C, u)
print('kernel K:', ssm_kernel(A, B, C, 5).round(3), '...')
assert np.allclose(y_rec, y_conv)
print('\u2713 recurrence and convolution give IDENTICAL outputs (the SSM duality)')

## 3 — O(L) SSM vs O(L²) attention

An SSM's cost grows linearly with sequence length; attention's grows quadratically. We count the
operations to make the scaling gap concrete.

In [ ]:
for L in [256, 1024, 4096, 16384]:
    attn = L * L          # all-pairs attention scores
    ssm = L               # one O(1) state update per token (recurrent inference)
    print(f'L={L:6d}:  attention ops ~ {attn:>12,}   SSM ops ~ {ssm:>8,}   ratio {attn/ssm:>8,.0f}x')
print('\nThe gap widens with length -> SSMs shine on very long sequences.')

## ✏️ Your turn

**Exercise.** Implement `ssm_step(A, B, C, h, u_t)` returning `(new_state, output)` for one
recurrence step, and `ssm_kernel_k(A, B, C, k)` returning the k-th convolution kernel coefficient
`C·A^k·B`. These are the two faces of the same linear SSM.

In [ ]:
def ssm_step(A, B, C, h, u_t):
    # TODO(you): return (new_state, output) = (A*h + B*u_t, C*new_state)
    return ...

def ssm_kernel_k(A, B, C, k):
    # TODO(you): the k-th kernel coefficient C * A^k * B
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
h, y0 = ssm_step(A, B, C, 0.0, u[0])
assert np.isclose(h, A*0.0 + B*u[0]) and np.isclose(y0, C*h)
assert np.isclose(ssm_kernel_k(A, B, C, 0), C*B)        # k=0 -> C*B
assert np.isclose(ssm_kernel_k(A, B, C, 2), C*A*A*B)
# rolling ssm_step matches the reference recurrence
hh, out = 0.0, []
for ut in u:
    hh, yt = ssm_step(A, B, C, hh, ut); out.append(yt)
assert np.allclose(out, y_rec)
print('\u2713 recurrence step and kernel coefficient are correct')

<details>
<summary>Solution</summary>

```python
def ssm_step(A, B, C, h, u_t):
    new_state = A * h + B * u_t
    return new_state, C * new_state

def ssm_kernel_k(A, B, C, k):
    return C * (A ** k) * B
```

The recurrence (constant memory, O(1)/token) is used for generation; the kernel (a parallel
convolution) is used for training. S4 chooses A cleverly for long memory; Mamba makes B, C depend on
the input for selectivity, trading the convolution for a parallel scan.

</details>